# Stage 01 — Data Ingestion
Prototype notebook for the data ingestion component.
Run cells top to bottom once, then copy the final code to `src/`.

In [60]:
import os
#os.chdir("/home/abood/Desktop/ML_Model_Monitoring_System_for_Credit_Risk")
print("Working directory:", os.getcwd())

Working directory: /home/abood/Desktop/ML_Model_Monitoring_System_for_Credit_Risk


In [61]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [62]:
from src.creditrisk.constants import *
from src.creditrisk.utils import read_yaml, create_directories, logger

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        return DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

In [63]:
import os
import urllib.request as request
from src.creditrisk.utils import logger

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self):
        """
        Downloads the train CSV directly from GitHub using urllib (no auth needed).
        """
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=str(self.config.source_URL),
                filename=str(self.config.local_data_file)
            )
            logger.info(f"File downloaded successfully and saved at: {filename}")
        else:
            logger.info(f"File already exists at: {self.config.local_data_file}")

    def extract_zip_file(self):
        """
        No-op — the file is already a plain CSV, nothing to extract.
        Just validates the file exists.
        """
        if os.path.exists(str(self.config.local_data_file)):
            logger.info(f"Data file ready at: {self.config.local_data_file}")
        else:
            raise FileNotFoundError(
                f"Expected data file not found at: {self.config.local_data_file}"
            )


In [64]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-05-04 19:28:19,651: INFO: common]: yaml file: config/config.yaml loaded successfully
[2026-05-04 19:28:19,653: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-04 19:28:19,656: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-05-04 19:28:19,657: INFO: common]: created directory at: artifacts
[2026-05-04 19:28:19,657: INFO: common]: created directory at: artifacts/data_ingestion
[2026-05-04 19:28:19,657: INFO: 2684012353]: File already exists at: artifacts/data_ingestion/train.csv
[2026-05-04 19:28:19,658: INFO: 2684012353]: Data file ready at: artifacts/data_ingestion/train.csv
